# EDA — Code Vulnerability Security DPO

**Dataset:** [`CyberNative/Code_Vulnerability_Security_DPO`](https://huggingface.co/datasets/CyberNative/Code_Vulnerability_Security_DPO)

Pares DPO sintéticos (generados con deepseek-coder-33b): para cada tarea hay una respuesta con código seguro (`chosen`) y otra vulnerable (`rejected`).

**Formato:** JSON Lines (con extensión `.json`). Columnas: `lang`, `vulnerability`, `system`, `question`, `chosen`, `rejected`.

**Preguntas guía para el EDA**
- ¿Cuántos pares hay por lenguaje?
- ¿Qué tipos de vulnerabilidad aparecen y cuánto se repiten?
- ¿`chosen` y `rejected` difieren en largo? (sesgo que el modelo podría aprender)
- ¿Hay preguntas repetidas o pares casi idénticos?

In [ ]:
import sys, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Busca la raíz del repo (donde está src/profiler.py) y la agrega al path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "profiler.py").exists())
sys.path.insert(0, str(ROOT))
from src.profiler import *

sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

ctx = eda_context("03")   # rutas: datos en data/raw/..., resultados en esta carpeta
print(ctx)

## 1. Inventario de archivos

In [ ]:
files = list_files(ctx.data_dir)
display(summarize_extensions(files))
display(files)

## 2. Cargar

In [ ]:
PATH = ctx.data_dir / "secure_programming_dpo.json"
df = load_table(PATH)
print(df.shape)
display(df.head(3))
display(profile_df(df))

## 3. Lenguajes y vulnerabilidades

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
df["lang"].value_counts().plot.bar(ax=ax); ax.set_title("Pares por lenguaje")
save_fig(fig, ctx, "pares_por_lenguaje"); plt.show()

print("Descripciones de vulnerabilidad únicas:", df["vulnerability"].nunique(), "de", len(df))
display(df["vulnerability"].value_counts().head(15))
print("Filas con system vacío:", (df["system"].fillna("").str.strip() == "").sum())

## 4. Largo de `chosen` vs `rejected`

In [ ]:
L = text_lengths(df, ["question", "chosen", "rejected"])
L["lang"] = df["lang"]
display(L.describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
L.melt(id_vars="lang", value_vars=["chosen", "rejected"]).pipe(
    lambda d: sns.boxplot(data=d, x="lang", y="value", hue="variable", ax=axes[0], showfliers=False))
axes[0].set_title("Largo del código por lenguaje"); axes[0].tick_params(axis="x", rotation=45)
sns.histplot(L["chosen"] - L["rejected"], bins=60, ax=axes[1])
axes[1].set_title("Diferencia de largo (chosen − rejected)")
plt.tight_layout(); save_fig(fig, ctx, "largo_chosen_rejected"); plt.show()
print(f"% de pares donde chosen es más largo: {(L.chosen > L.rejected).mean():.1%}")

## 5. Duplicados

In [ ]:
display(duplicates_report(df, {"question": ["question"], "chosen": ["chosen"],
                               "rejected": ["rejected"], "chosen == rejected": ["chosen", "rejected"]}))
print("Pares con chosen idéntico a rejected:", (df["chosen"] == df["rejected"]).sum())

## 6. Ejemplos

In [ ]:
show_examples(df, n=2, cols=["lang", "vulnerability", "question", "chosen", "rejected"], max_chars=700)

## Guardar resultados
Se guardan en `tables/` de esta carpeta. `resumen.csv` lo consolida `eda/00_general`.

In [ ]:
save_table(profile_df(df), ctx, "perfil_columnas")
display(save_summary(ctx, "CyberNative/Code_Vulnerability_Security_DPO", df, total_rows=None, dup_cols=["question"],
                     notas=""))

## Próximos pasos / notas
- Anota tus hallazgos en el `README.md` de esta carpeta.
-